# Unit II, Homework 1: Vectorization

**CS 301, Introduction to Data Science, NJIT.** Unit II, Homework 1. Handed in on Canvas, as this notebook, by the due date given there.

Three parts, 20 points. **A, read (6):** six pieces of code; run each, then say in plain words what it computes. **B, recognize (6):** eight short questions, answered in a variable. **C, write (8):** four functions without loops, each with a check.

**Purpose.** The computations of this unit are written as operations on whole arrays, in NumPy and in PyTorch, without loops in Python. This homework checks that you can read such code and say what it computes, with the shapes; that you recognize the shapes, the axes and the broadcasting involved; and that you can write it. The rest of the unit and the exam require all three.

## How this works

- Work alone. You may ask an AI assistant in Part C; if you do, keep the prompt in the cell marked **Your prompt**, at the end.
- First choose **Save a copy in Drive** in the **File** menu. Run the cells in order.
- <font color="#c00000" size="5"><b>Your turn</b></font> marks what you write: a description in Part A, a value in Part B, code in Part C. A cell under the word **Check** tests your code and says what to look at when it fails.
- Hand in: run every cell from the top, then **File → Download → Download .ipynb**, and upload the file on Canvas. The outputs are part of what is read.

In [ ]:
# Setup. Run this cell first.
import ast
import inspect
import io
import textwrap
import time
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

SITE = "https://ikoutis.github.io/cs301-labs/unit-2/"
IMAGES = SITE + "images/"


def read_image(file):
    """An image file as a NumPy array with values from 0 to 1."""
    return np.array(Image.open(urlopen(IMAGES + file))) / 255.0


def show(pictures, titles):
    """Arrays as pictures: two dimensions in grey, three in colour."""
    fig, axes = plt.subplots(1, len(pictures), figsize=(5 * len(pictures), 4.2), squeeze=False)
    for ax, picture, title in zip(axes.flat, pictures, titles):
        ax.imshow(np.clip(picture, 0, 1), cmap="gray", vmin=0, vmax=1)
        ax.set_title(title)
    fig.tight_layout()
    plt.show()


def average_time(function, data, repeats=5):
    """The average time of one call function(data), in seconds."""
    total = 0.0
    for _ in range(repeats):
        start = time.perf_counter()
        function(data)
        total += time.perf_counter() - start
    return total / repeats


def has_loop(function):
    """True when the text of the function contains a loop or a comprehension."""
    try:
        tree = ast.parse(textwrap.dedent(inspect.getsource(function)))
    except (OSError, TypeError, SyntaxError):
        return False
    loops = (ast.For, ast.While, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    return any(isinstance(node, loops) for node in ast.walk(tree))


def reference_colours(X, K=5, rounds=10):
    """K colours that occur often among the rows of X, darkest first (the method k-means)."""
    grey_weights = np.array([0.299, 0.587, 0.114])
    sample = X[::7]
    order = np.argsort(sample @ grey_weights)
    C = np.array([sample[part].mean(axis=0) for part in np.array_split(order, K)])
    for _ in range(rounds):
        distance = ((sample[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
        nearest = distance.argmin(axis=1)
        for j in range(K):
            if (nearest == j).any():
                C[j] = sample[nearest == j].mean(axis=0)
    return C[np.argsort(C @ grey_weights)]


IMAGE = 7                                  # any number from 1 to 30: the image of that group in meeting 10
images = pd.read_csv(IMAGES + "images.csv", index_col="number")
image = read_image(images.loc[IMAGE, "file"])
height, width = image.shape[0], image.shape[1]
X = image.reshape(-1, 3)                   # the pixels: n points with m = 3 coordinates
n, m = X.shape
K = 5
C = reference_colours(X, K)                # the reference colours: K rows, 3 columns
W = C.copy()                               # the layer of meeting 11: one neuron for each colour
b = -0.5 * (C ** 2).sum(axis=1)
S = X @ W.T + b                            # the scores: n rows, K columns
batch = np.load(io.BytesIO(urlopen(IMAGES + "batch-96.npy").read()))      # thirty small images
weather = pd.read_csv(SITE + "weather/newark-weather.csv").dropna(subset=["tmax_f", "tmin_f"])
highs = weather["tmax_f"].to_numpy()       # the high of every day at Newark airport, 2006 to 2024
rng = np.random.default_rng(0)
print(f"image {IMAGE}: {image.shape}   X: {X.shape}   W: {W.shape}   b: {b.shape}   S: {S.shape}")
print(f"batch: {batch.shape}   highs: {highs.shape}")

## <font color="#1967d2"><b>Part A: Read, run, describe</b></font> (6 points, 1 each)

For each piece of code: run it, read the output, then describe in two or three sentences what it computes, with the shapes. Say what the operation does to the whole array, not what each line is called.

In [ ]:
w_grey = np.array([0.299, 0.587, 0.114])
grey = image @ w_grey
print("image:", image.shape, "  w_grey:", w_grey.shape, "  grey:", grey.shape)
show([image, grey], ["image", "image @ w_grey"])

<font color="#c00000" size="5"><b>Your turn.</b></font> **A1.** What does `image @ w_grey` compute? Over which dimension of `image` does the product run, and why does the result have two dimensions?

*Your description here.*

In [ ]:
D = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)
nearest = D.argmin(axis=1)
print("X[:, None, :]:", X[:, None, :].shape, "  C[None, :, :]:", C[None, :, :].shape)
print("the difference:", (X[:, None, :] - C[None, :, :]).shape, "  D:", D.shape, "  nearest:", nearest.shape)
print("the first five points:", nearest[:5])
show([image, C[nearest].reshape(height, width, 3)], ["image", "C[nearest]"])

<font color="#c00000" size="5"><b>Your turn.</b></font> **A2.** What does `D` hold, entry by entry? What does `nearest` hold, and what is the picture on the right? Say where broadcasting happens.

*Your description here.*

In [ ]:
P = X[:, None, :] * W[None, :, :]
S_slow = P.sum(axis=2) + b
print("P:", P.shape, f"{P.nbytes / 1e6:.0f} MB", "  S_slow:", S_slow.shape, "  S:", S.shape, f"{S.nbytes / 1e6:.1f} MB")
print("the same numbers as X @ W.T + b:", np.allclose(S_slow, S))
t_slow = average_time(lambda X: (X[:, None, :] * W[None, :, :]).sum(axis=2) + b, X)
t_fast = average_time(lambda X: X @ W.T + b, X)
print(f"P.sum(axis=2) + b: {t_slow:.4f} s   X @ W.T + b: {t_fast:.4f} s")

<font color="#c00000" size="5"><b>Your turn.</b></font> **A3.** What does `P[i, j, c]` hold? Why do the two lines give the same `S`, and why is `X @ W.T + b` the better way?

*Your description here.*

In [ ]:
bright = image.mean(axis=2) > 0.5
new = image.copy()
new[bright] = 1 - new[bright]
print("bright:", bright.shape, bright.dtype, f"  True on {bright.mean():.1%} of the pixels")
show([image, new], ["image", "new"])

<font color="#c00000" size="5"><b>Your turn.</b></font> **A4.** What is `bright`, and what does `new[bright] = 1 - new[bright]` do to the picture? Which pixels are left as they were?

*Your description here.*

In [ ]:
winner = S.argmax(axis=1)
counts = np.bincount(winner, minlength=K)
print("winner:", winner.shape, "  counts:", counts, "  counts.sum() == n:", counts.sum() == n)
print("share of each neuron:", np.round(counts / n, 3))

<font color="#c00000" size="5"><b>Your turn.</b></font> **A5.** What does `winner` hold for one pixel, and what does `counts` count? Why must the counts add to n?

*Your description here.*

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
X_t, W_t, b_t = [torch.from_numpy(a).float().to(device) for a in (X, W, b)]
S_t = X_t @ W_t.T + b_t
winner_t = S_t.argmax(dim=1).cpu().numpy()
print("device:", device, "  S_t:", tuple(S_t.shape), S_t.dtype, "  on", S_t.device)
print("same winners as NumPy:", (winner_t == winner).mean())

<font color="#c00000" size="5"><b>Your turn.</b></font> **A6.** What happens to the three arrays in the second line? Where does `S_t` live? Why `.cpu()` before `.numpy()`, and what does the last line test?

*Your description here.*

## <font color="#1967d2"><b>Part B: Recognize</b></font> (6 points, 0.75 each)

Replace the `None` in each cell with your answer. A shape is a tuple, such as `(500, 6)`; a choice is a letter in quotes, such as `"b"`. The cell checks only the form of the answer.

<font color="#c00000" size="5"><b>Your turn.</b></font> **B1.** `X` has shape (500, 8), `W` has shape (6, 8) and `b` has shape (6,). What is the shape of `X @ W.T + b`?

In [ ]:
ANSWER_B1 = None        # a shape, for example (500, 6)

assert isinstance(ANSWER_B1, tuple) and all(isinstance(v, int) for v in ANSWER_B1), "ANSWER_B1 must be a tuple of integers, such as (500, 6)."
print("B1: a shape of", len(ANSWER_B1), "dimensions.")

<font color="#c00000" size="5"><b>Your turn.</b></font> **B2.** `A` has shape (4, 7, 3). What is the shape of `A.sum(axis=1)`?

In [ ]:
ANSWER_B2 = None        # a shape, for example (500, 6)

assert isinstance(ANSWER_B2, tuple) and all(isinstance(v, int) for v in ANSWER_B2), "ANSWER_B2 must be a tuple of integers, such as (500, 6)."
print("B2: a shape of", len(ANSWER_B2), "dimensions.")

<font color="#c00000" size="5"><b>Your turn.</b></font> **B3.** Which line uses broadcasting?  a) `X @ w`   b) `X - X.mean(axis=0)`   c) `X.T`   d) `X.reshape(-1)`

In [ ]:
ANSWER_B3 = None        # a letter in quotes, for example "b"

assert ANSWER_B3 in ("a", "b", "c", "d"), 'ANSWER_B3 must be one letter in quotes: "a", "b", "c" or "d".'
print("B3: answered.")

<font color="#c00000" size="5"><b>Your turn.</b></font> **B4.** Four ways to compute the grey image. Which is fastest?  a) two loops over the pixels, with the three products in the inner loop   b) one loop over the rows, with `row @ w_grey` for each row   c) `image @ w_grey`   d) a list comprehension over the pixels

In [ ]:
ANSWER_B4 = None        # a letter in quotes, for example "b"

assert ANSWER_B4 in ("a", "b", "c", "d"), 'ANSWER_B4 must be one letter in quotes: "a", "b", "c" or "d".'
print("B4: answered.")

<font color="#c00000" size="5"><b>Your turn.</b></font> **B5.** `S` has shape (1000, 4). What is the shape of `S.max(axis=1, keepdims=True)`?

In [ ]:
ANSWER_B5 = None        # a shape, for example (500, 6)

assert isinstance(ANSWER_B5, tuple) and all(isinstance(v, int) for v in ANSWER_B5), "ANSWER_B5 must be a tuple of integers, such as (500, 6)."
print("B5: a shape of", len(ANSWER_B5), "dimensions.")

<font color="#c00000" size="5"><b>Your turn.</b></font> **B6.** `X` has shape (n, 3) and `W` has shape (5, 3), one neuron per row. `X @ W`:  a) raises an error   b) runs, with wrong numbers   c) gives the scores

In [ ]:
ANSWER_B6 = None        # a letter in quotes, for example "b"

assert ANSWER_B6 in ("a", "b", "c", "d"), 'ANSWER_B6 must be one letter in quotes: "a", "b", "c" or "d".'
print("B6: answered.")

<font color="#c00000" size="5"><b>Your turn.</b></font> **B7.** `X` has shape (n, 3) and `W` has shape (3, 3), one neuron per row. `X @ W + b`:  a) raises an error   b) runs, with wrong numbers   c) gives the scores

In [ ]:
ANSWER_B7 = None        # a letter in quotes, for example "b"

assert ANSWER_B7 in ("a", "b", "c", "d"), 'ANSWER_B7 must be one letter in quotes: "a", "b", "c" or "d".'
print("B7: answered.")

<font color="#c00000" size="5"><b>Your turn.</b></font> **B8.** Two measurements add the same number of pairs of numbers: a) two arrays of 16 million numbers, four times; b) two arrays of 16 thousand numbers, 4,096 times. Which one is memory-bound?

In [ ]:
ANSWER_B8 = None        # a letter in quotes, for example "b"

assert ANSWER_B8 in ("a", "b", "c", "d"), 'ANSWER_B8 must be one letter in quotes: "a", "b", "c" or "d".'
print("B8: answered.")

## <font color="#1967d2"><b>Part C: Write</b></font> (8 points, 2 each)

Each function is one or two lines on whole arrays: no `for`, no `while`, no comprehension. The check runs it and compares it with a slow version.

<font color="#c00000" size="5"><b>Your turn.</b></font> **C1. Standardize the rows.** `standardize_rows(X)` returns an array of the same shape in which every **row** has mean 0 and standard deviation 1 (divide by the number of entries, as `std` does). The labs standardized the columns, with `axis=0`; here it is the rows.

In [ ]:
def standardize_rows(X):
    """Every row of X brought to mean 0 and standard deviation 1."""
    return ...      # Your turn: one line, with axis=1 and keepdims=True

**Check.**

In [ ]:
R = rng.random((6, 40)) * 5 + 2
try:
    Z = standardize_rows(R)
except ValueError:
    raise AssertionError("The shapes do not fit: the mean and the deviation of the rows must keep the shape (6, 1) "
                         "to be subtracted from the rows of R. Use keepdims=True.") from None
assert Z is not ..., "Replace the three dots."
assert not has_loop(standardize_rows), "No loop: use operations on the whole array."
assert np.asarray(Z).shape == R.shape, f"The result must keep the shape {R.shape}."
assert np.allclose(Z.mean(axis=1), 0) and np.allclose(Z.std(axis=1), 1), (
    "Every row must have mean 0 and standard deviation 1: take the mean and the deviation along axis=1, and keep them as a column (keepdims=True).")
print("C1 correct.")

<font color="#c00000" size="5"><b>Your turn.</b></font> **C2. The nearest colour of every pixel.** `nearest_colours(X, C)` returns an array of n integers: for each row of `X`, the index of the row of `C` nearest to it. Two ways work without a loop: the distances of A2, or the layer of meeting 11, whose scores `X @ C.T - 0.5 * (C ** 2).sum(axis=1)` are largest for the nearest colour.

In [ ]:
def nearest_colours(X, C):
    """For each row of X, the index of the nearest row of C."""
    return ...      # Your turn: no loop

**Check.** The slow version visits every pair of a point and a colour.

In [ ]:
def nearest_loops(X, C):
    out = np.empty(len(X), dtype=int)
    for i in range(len(X)):
        best, best_d = 0, np.inf
        for j in range(len(C)):
            d = ((X[i] - C[j]) ** 2).sum()
            if d < best_d:
                best, best_d = j, d
        out[i] = best
    return out


idx = nearest_colours(X, C)
assert idx is not ..., "Replace the three dots."
assert not has_loop(nearest_colours), "No loop: use broadcasting, or the layer's scores."
idx = np.asarray(idx)
assert idx.shape == (n,), f"The result must have shape {(n,)}: one index for each point."
assert np.array_equal(idx[:500], nearest_loops(X[:500], C)), "The indices differ from the slow version. Check the axis of the sum and of the argmin, or the bias of the layer."
print("C2 correct.")
show([image, C[idx].reshape(height, width, 3)], ["image", "its nearest colours"])

<font color="#c00000" size="5"><b>Your turn.</b></font> **C3. A layer on a whole batch.** `batch` holds thirty images of 96 by 96 pixels: shape (30, 96, 96, 3). `layer_on_batch(B, W, b)` returns the scores of the K neurons on every pixel of every image, shape (30, 96, 96, K), with **one** `@` and no `reshape`: `@` treats the last two dimensions as the matrix and the dimensions before them as a batch. Then write the same function for PyTorch tensors.

In [ ]:
def layer_on_batch(B, W, b):
    """The scores of the layer (W, b) on every pixel of every image of the batch B: shape (N, H, W, K)."""
    return ...      # Your turn: one @, no reshape


def layer_on_batch_torch(B_t, W_t, b_t):
    """The same, for PyTorch tensors."""
    return ...      # Your turn

**Check.**

In [ ]:
try:
    S_batch = layer_on_batch(batch, W, b)
except ValueError:
    raise AssertionError("The shapes do not fit: B has shape (N, H, W, 3) and W has shape (K, 3). "
                         "Which array has to be transposed?") from None
assert S_batch is not ..., "Replace the three dots."
assert not has_loop(layer_on_batch), "No loop."
assert "reshape" not in inspect.getsource(layer_on_batch), "No reshape: @ handles the batch by itself."
assert np.asarray(S_batch).shape == (30, 96, 96, K), f"The result must have shape {(30, 96, 96, K)}."
expected = (batch.reshape(-1, 3) @ W.T + b).reshape(30, 96, 96, K)
assert np.allclose(S_batch, expected), "The numbers differ from the scores of the layer: which array is transposed, and is b added?"
B_t, W_t, b_t = torch.from_numpy(batch).float(), torch.from_numpy(W).float(), torch.from_numpy(b).float()
S_batch_t = layer_on_batch_torch(B_t, W_t, b_t)
assert isinstance(S_batch_t, torch.Tensor), "layer_on_batch_torch must return a tensor."
assert tuple(S_batch_t.shape) == (30, 96, 96, K), f"The tensor must have shape {(30, 96, 96, K)}."
assert np.allclose(S_batch_t.numpy(), expected, atol=1e-4), "The tensor's numbers differ from the scores (float32 against float64 is allowed)."
print("C3 correct.")

<font color="#c00000" size="5"><b>Your turn.</b></font> **C4. Vectorize a loop, and measure.** `rises_loops(highs, step)` counts the days whose high was more than `step` degrees above the high of the day before. Write `rises(highs, step)` without a loop: the difference between every day and the day before is one subtraction of two slices of `highs`. Then the check measures both.

In [ ]:
def rises_loops(highs, step):
    """The number of days whose high is more than `step` degrees above the high of the day before."""
    count = 0
    for i in range(1, len(highs)):
        if highs[i] - highs[i - 1] > step:
            count += 1
    return count


print("days that rose by more than 10 degrees:", rises_loops(highs, 10), "of", len(highs))

In [ ]:
def rises(highs, step):
    """The same count, without a loop."""
    return ...      # Your turn: two slices of highs, a comparison, a sum

**Check.** Then the two versions are timed; `FACTOR_C4` is how many times faster yours is.

In [ ]:
assert rises(highs, 10) is not ..., "Replace the three dots."
assert not has_loop(rises), "No loop: subtract highs[:-1] from highs[1:]."
for step in (5, 10, 15):
    assert int(rises(highs, step)) == rises_loops(highs, step), f"rises(highs, {step}) differs from the loop: check the comparison and which slice comes first."
t_loops = average_time(lambda h: rises_loops(h, 10), highs, 3)
t_fast = average_time(lambda h: rises(h, 10), highs, 20)
FACTOR_C4 = t_loops / t_fast
print(f"C4 correct. Loop {t_loops * 1e3:.2f} ms, vectorized {t_fast * 1e3:.3f} ms: {FACTOR_C4:.0f} times faster.")

<font color="#c00000" size="5"><b>Your prompt.</b></font> If you asked an AI assistant for any part of C, paste the prompts here.

*Your prompts here, or 'none'.*

## Hand in

Run every cell from the top (**Runtime → Run all**) and look at the outputs: every check prints *correct*, and every Part A cell has your description. Then **File → Download → Download .ipynb**, and upload the file on Canvas, in *Unit II - Homework 1: Vectorization*, by the due date given there.